# GFTM with an (emulated) width search vs GS2: does a search alone make GFTM over-predict? (Fusion_PhD-qp5.46)

Advisory numbers only. The argument tested: TGLF's width search over-predicts $\gamma$ (`tglf_sat1_sat2`), but converged TGLF also over-predicts at a
fixed width. If a max-$\gamma$ width search brings over-prediction back in GFTM, the search itself causes it.

**GFTM's own `FIND_WIDTH` search is dead code in the production build** (`gftm_max.f90` is not in the Makefile and every call is commented out;
line numbers in the README), so the search is **emulated from the fixed-width leaves (grid search, no bisection)**:
per sample, $W_{\rm sel}=\arg\max_{W\in[0.3,1.65]}\gamma_0(W;\,N_{\rm min})$ over the 24-point `WIDTH_ORACLE_GRID` (dominant mode of any sign of $\omega$, as the search's max-$\gamma$ criterion),
then $\gamma,\omega,\theta_\phi$ at $N_{\rm max}$ and $W_{\rm sel}$. Arms (NBASIS_MIN, NBASIS_MAX) = (2,4), (4,8), (16,16), (32,32); FILTER 0.5 and 0.
Fields stay on in the search (EM), as TGLF SAT1 and as the dormant GFTM code.

Table format, populations and plots are those of `tglf_sat1_sat2`: $r=\log_{10}(\gamma_{\rm code}/\gamma_{\rm GS2})$ over samples with $\gamma_{\rm code}>0$, found = $\gamma>0$.
Leaves: `GFTM/Runs/LATIN_HYPERCUBE/<case>/kbm_free/galF[_f0p0]_<w>_nb<N>_nx<X>` (FIELD_GALERKIN F), NXGRID 32 up to NBASIS 16, 40 at 32.

In [ ]:
import os
for threads in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[threads] = "1"
import json
import re
from multiprocessing import get_context
from pathlib import Path
import warnings
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import NullLocator, StrMethodFormatter
import pyrokinetics
from pyrokinetics import PyroHypercube
from pyrokinetics.diagnostics.extent import Extent
from general_analysis import mode_classification as mc

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir())
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)
warnings.filterwarnings("ignore")
print("pyrokinetics", pyrokinetics.__version__, pyrokinetics.__file__)

data_root = Path(os.path.expandvars(os.environ["GK_DATA_ROOT"]))     # WORK: the GS2 kbm_free cubes
run_root = Path(os.path.expandvars("$GYRO_DATA_OUTPUT"))             # scratch: the new leaves (copied to WORK after the runs)
plots_dir = ROOT / "Plots" / "TGLF_GFTM_KBM_PAPER" / "gftm_width_search"
csv_dir = Path("csv")   # small summaries only
plots_dir.mkdir(parents=True, exist_ok=True); csv_dir.mkdir(exist_ok=True)
cases = ["SPR-045", "M1", "R4", "R3"]
widths = [0.2, 0.2297, 0.2639, 0.3031, 0.3482, 0.4, 0.4595, 0.5279, 0.6063, 0.6965, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6,
          1.9218, 2.3083, 2.7726, 3.3302, 4.0]    # registry.WIDTH_ORACLE_GRID
width_tag = lambda width: f"w{width:.4f}".replace(".", "p")
gftm_cells = {2: 32, 4: 32, 8: 32, 16: 32, 32: 40}   # NBASIS: NXGRID (>= NBASIS + 8)
gftm_filters = {"": 0.5, "_f0p0": 0.0}
search_arms = [(2, 4), (4, 8), (16, 16), (32, 32)]   # (NBASIS_MIN, NBASIS_MAX)
width_window = (0.3, 1.65)   # TGLF's search range, WIDTH_MIN to WIDTH
metrics = ["gamma", "omega", "phi", "apar"]
n_cpu = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))
rng = np.random.default_rng(0)

## GS2 reference and population
The user's classifier on each `kbm_free` cube (as `results/kbm_fixed_width_nbasis/runs/population.py`, which built the leaves). GS2's
`phi_squared`/`apar_squared` sit on a union $\theta$ grid NaN-padded per sample, so the extent is taken per sample on its finite points.

In [ ]:
def tied_species(sample):
    # GyroRun's links for the kbm_free cubes (not stored in pyroscan.json): ions share ion1's a/L_T, every a/L_n is the electron's
    species = gs2_scan.sample_pyro(sample).local_species
    ions = [name for name in species.names if name != "electron"]
    for ion in ions[1:]:
        species[ion].inverse_lt = species[ions[0]].inverse_lt
    for ion in ions:
        species[ion].inverse_ln = species["electron"].inverse_ln
    species.enforce_quasineutrality("electron")
    return {name: (float(species[name].temp.m), float(species[name].dens.m), float(species[name].inverse_lt.m)) for name in species.names}


def extent_rows(power):
    # pyro's 95% rule per sample on its own finite theta points; power: (sample, theta) DataArray
    if np.isfinite(power.values).all():
        return Extent._compute(power, 0.95)[0].values
    return np.array([float(Extent._compute(row.dropna("theta").expand_dims("sample"), 0.95)[0][0]) if np.isfinite(row.values).any() else np.nan
                     for row in power])


gs2 = {}
for case in cases:
    leaf = data_root / "GS2/Runs/LATIN_HYPERCUBE" / case / "kbm_free"
    gs2_scan = PyroHypercube(pyroscan_json=leaf / "pyroscan.json", load_base_pyro=True)
    gs2_scan.load_gk_output(netcdf_file=leaf / "pyroscan.nc")
    gs2_output = gs2_scan.gk_output.data
    names = np.array([str(name) for name in gs2_output.sample_name.values])
    with get_context("fork").Pool(n_cpu) as pool:
        species = pool.map(tied_species, range(len(names)), chunksize=8)
    heat = gs2_output.heat.sum("field").pint.dequantify()
    particle = gs2_output.particle.sum("field").pint.dequantify()

    def chi(group):   # (Q_s - 1.5 T_s Gamma_s) / (n_s T_s a/L_Ts) summed over the group, as mode_classification.indicators
        numerator = denominator = 0.0
        for name in group:
            temp, dens, inverse_lt = np.array([sp[name] for sp in species]).T
            numerator = numerator + heat.sel(species=name).values - 1.5 * temp * particle.sel(species=name).values
            denominator = denominator + dens * temp * inverse_lt
        return numerator / denominator
    ions = [str(s) for s in gs2_output.species.values if s != "electron"]
    with np.errstate(divide="ignore", invalid="ignore"):
        labels = np.asarray(mc.classify(dict(T=gs2_output.tearing_parameter.values.ravel(),
                                             omega=mc.ION_DIRECTION * gs2_output.mode_frequency.pint.dequantify().values.ravel(),
                                             chi_ratio=chi(["electron"]) / chi(ions))))
    gamma = gs2_output.growth_rate.pint.dequantify().values.ravel()
    tolerance = gs2_output.growth_rate_tolerance.pint.dequantify().values.ravel()
    kbm = np.isfinite(gamma) & (gamma > 0) & (tolerance < 0.05) & (labels == "KBM")
    selected = gs2_output.set_index(sample="sample_name").sel(sample=names[kbm])
    gs2[case] = dict(names=names[kbm], gamma=gamma[kbm], omega=mc.ION_DIRECTION * selected.mode_frequency.pint.dequantify().values.ravel(),
                     phi=extent_rows(selected.phi_squared.pint.dequantify().squeeze("kx").transpose("sample", "theta")),
                     apar=extent_rows(selected.apar_squared.pint.dequantify().squeeze("kx").transpose("sample", "theta")),
                     units=re.sub(r"_pyroscan_base\d+", "", str(gs2_output.growth_rate.data.units)))
    print(case, "GS2-KBMs", kbm.sum(), "| median GS2 extent phi", np.median(gs2[case]["phi"]), "apar", np.median(gs2[case]["apar"]), flush=True)
    del gs2_output, selected
reference = pd.concat([pd.DataFrame(dict(case=case, sample=gs2[case]["names"], **{f"{m}_gs2": gs2[case][m] for m in metrics})) for case in cases])
assert len(reference) == 738, len(reference)

## GFTM fixed-width leaves
One `PyroHypercube` per leaf; settings from `manifest.json` checked against the leaf name. Dominant mode = mode 0 (IBRANCH $-1$), checked by `argmax`.

In [ ]:
def load_leaf(spec):
    case, filter_value, nbasis, nxgrid, width, leaf = spec
    if not (leaf / "pyroscan.nc").is_file():
        return None
    scan = PyroHypercube(pyroscan_json=leaf / "pyroscan.json", load_base_pyro=True)
    scan.load_gk_output(netcdf_file=leaf / "pyroscan.nc")
    flags = json.loads((leaf / "manifest.json").read_text())["code_flags"]
    assert (float(flags["WIDTH"]), float(flags["FILTER"]), int(flags["NBASIS_MAX"]), int(flags["NXGRID"])) == (width, filter_value, nbasis, nxgrid), leaf
    assert not flags["FIND_WIDTH"] and not flags["FIELD_GALERKIN"], leaf
    output = scan.gk_output.data
    assert re.sub(r"_pyroscan_base\d+", "", str(output.growth_rate.data.units)) == gs2[case]["units"], "normalisations differ"
    names = [str(name) for name in output.sample_name.values]
    output = output.set_index(sample="sample_name")
    growth = output.growth_rate.pint.dequantify().transpose("sample", "mode")
    assert (growth.fillna(-np.inf).argmax("mode") == 0).all(), leaf
    gamma = growth.isel(mode=0).fillna(0).clip(min=0).values   # mode0-guard-allow: IBRANCH -1, argmax==0 asserted above
    power = (np.abs(output.phi.isel(mode=0).pint.dequantify().squeeze("kx")) ** 2).transpose("sample", "theta")  # mode0-guard-allow: as above
    return pd.DataFrame(dict(case=case, filter=filter_value, nbasis=nbasis, nxgrid=nxgrid, width=width, sample=names, gamma=gamma,
                             phi=np.where(gamma > 0, extent_rows(power), np.nan)))


specs = [(case, value, nbasis, nxgrid, width, run_root / "GFTM/Runs/LATIN_HYPERCUBE" / case / "kbm_free" / f"galF{tag}_{width_tag(width)}_nb{nbasis}_nx{nxgrid}")
         for case in cases for tag, value in gftm_filters.items() for nbasis, nxgrid in gftm_cells.items() for width in widths]
with get_context("fork").Pool(n_cpu) as pool:
    tables = [table for table in pool.imap_unordered(load_leaf, specs, chunksize=4) if table is not None]
runs = pd.concat(tables, ignore_index=True)
print(runs.groupby(["filter", "nbasis"]).size().to_string())
# every (case, filter, nbasis, width, GS2-KBM sample) must exist: a missing row would be silently read as no-search-hit
assert (runs.groupby(["filter", "nbasis"]).size() == len(reference) * len(widths)).all(), "incomplete leaf set"
runs = runs.merge(reference[["case", "sample", "gamma_gs2", "phi_gs2"]], on=["case", "sample"], validate="many_to_one")

## Emulated search
At $N_{\rm min}$: the grid width in $[0.3,1.65]$ with the largest $\gamma_0$ (first on a tie; a sample with $\gamma_0=0$ at every width finds nothing and is a miss).
At $N_{\rm max}$: $\gamma$ and the $\phi$ extent at that width. `fixed 0.4` is the same table at $W=0.4$, $N=N_{\rm max}$.

In [ ]:
plot_dir = plots_dir
sample_order = {(row.case, row.sample): i for i, row in enumerate(reference.itertuples())}


def finish(table):
    table = table.assign(order=[sample_order[key] for key in zip(table["case"], table["sample"])]).sort_values("order").reset_index(drop=True)
    table["extent_ratio"] = table.phi / table.phi_gs2
    return table


arms = {}   # (kind, filter, nmin, nmax) -> per-sample table in reference order
for filter_value in gftm_filters.values():
    fixed = runs[runs["filter"] == filter_value]
    in_window = fixed[(fixed.width >= width_window[0]) & (fixed.width <= width_window[1])]
    for nmin, nmax in search_arms:
        low = in_window[in_window.nbasis == nmin].sort_values("width")
        best = low.loc[low.groupby(["case", "sample"]).gamma.idxmax()]
        best = best[best.gamma > 0][["case", "sample", "width"]].rename(columns={"width": "width_selected"})
        final = fixed[fixed.nbasis == nmax].merge(best, left_on=["case", "sample", "width"], right_on=["case", "sample", "width_selected"])
        final = reference[["case", "sample", "gamma_gs2", "phi_gs2"]].merge(final.drop(columns=["width", "gamma_gs2", "phi_gs2"]), on=["case", "sample"], how="left")
        final["gamma"] = final.gamma.fillna(0)
        arms["search", filter_value, nmin, nmax] = finish(final)
        fixed04 = fixed[(fixed.nbasis == nmax) & np.isclose(fixed.width, 0.4)].assign(width_selected=0.4)
        arms["fixed", filter_value, nmin, nmax] = finish(fixed04)
for key, table in arms.items():
    assert len(table) == 738 and table.gamma.notna().all(), key

## Bias table (the `tglf_sat1_sat2` statistic)

In [ ]:
def bias_stats(table):
    found = (table.gamma > 0).values
    ratio_log = np.log10(table.gamma.values[found] / table.gamma_gs2.values[found])
    boot = [np.median(rng.choice(ratio_log, ratio_log.size)) for _ in range(10000)]
    extent = table.extent_ratio.values[found]
    width = table.width_selected.values[found]
    return dict(n=len(table), n_found=int(found.sum()), n_miss=int((~found).sum()), frac_above=float((ratio_log > 0).mean()),
                median_log_ratio=float(np.median(ratio_log)), ci_lo=float(np.percentile(boot, 2.5)), ci_hi=float(np.percentile(boot, 97.5)),
                mean_ratio=float((table.gamma.values[found] / table.gamma_gs2.values[found]).mean()),
                logmed_abs_err=float(np.log10(np.median(np.abs(table.gamma.values - table.gamma_gs2.values)))),
                median_width=float(np.median(width)), frac_edge_low=float(np.isclose(width, 0.3031).mean()), frac_edge_high=float(np.isclose(width, 1.6).mean()),
                median_extent_ratio_phi=float(np.nanmedian(extent)))


rng = np.random.default_rng(0)
rows = []
for (kind, filter_value, nmin, nmax), table in arms.items():
    for case in (*cases, "pooled"):
        subset = table if case == "pooled" else table[table.case == case]
        rows.append(dict(kind=kind, filter=filter_value, nbasis_min=nmin, nbasis_max=nmax, case=case, **bias_stats(subset)))
summary = pd.DataFrame(rows)
summary.to_csv(csv_dir / "bias_by_case_setting.csv", index=False)
print(summary[summary.case == "pooled"].round(3).to_string())
print(summary[summary.case != "pooled"].round(3).to_string())

## Figures: the user's two TGLF plots (`tglf_sat1_sat2` cells 11 and 13), same code, GFTM search data
`pooled_arm` and `pooled_gs2` carry the same dict layout as in that notebook; only the data changes.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from matplotlib.ticker import LogLocator, NullFormatter
from scipy.stats import spearmanr

def pooled_arm(filter_value, nmin, nmax):
    table = arms["search", filter_value, nmin, nmax]
    return dict(gamma=table.gamma.values, extent=table.phi.values)

pooled_gs2 = dict(gamma=reference.gamma_gs2.values, extent=reference.phi_gs2.values)
pooled_case = reference.case.values
copied_dir = Path(os.path.expandvars("$WORK/fwatts/Papers/General_Analysis/Plots/TGLF_GFTM_KBM_PAPER/gftm_width_search"))
copied_dir.mkdir(parents=True, exist_ok=True)

# cell 13: gamma vs gamma_GS2, pooled, one FILTER, all NBASIS overlaid (blue = low NBASIS -> red = high)
low, high = 1e-3, 1e1   # axis limits; points outside are cut off (counted below), not removed from the data
bin_edges, min_per_bin = np.logspace(-2, 0.5, 11), 10
nbasis_cmap = LinearSegmentedColormap.from_list("blue_red", ["tab:blue", "tab:red"])
nbasis_colours = nbasis_cmap(np.linspace(0, 1, len(search_arms)))
for filter_value in gftm_filters.values():
    with plt.style.context([ROOT / "src/general_analysis/paper.mplstyle", ROOT.parent / "styles/paper.mplstyle"]):
        fig, ax = plt.subplots(figsize=(0.85 * 6.30, 0.6 * 6.30 * 0.85), layout="constrained")
        for (nb_min, nb_max), colour in zip(search_arms, nbasis_colours):
            arm = pooled_arm(filter_value, nb_min, nb_max)
            ok = (arm["gamma"] > 0) & np.isfinite(arm["extent"])
            x, y = pooled_gs2["gamma"][ok], arm["gamma"][ok]
            print(f"FILTER {filter_value:g} NBASIS {nb_min}-{nb_max}: {((x < low) | (y < low) | (x > high) | (y > high)).sum()} points outside axes, "
                  f"median gamma_G/gamma_GS2 = {np.median(y / x):.2f}")
            ax.scatter(x, y, s=10, alpha=0.2, color=colour, lw=0)
            which = np.digitize(x, bin_edges)
            centres, medians = zip(*[(np.sqrt(bin_edges[i - 1] * bin_edges[i]), np.median(y[which == i]))
                                     for i in range(1, len(bin_edges)) if (which == i).sum() >= min_per_bin])
            ax.plot(centres, medians, "-o", color=colour, lw=1.5, ms=3, mec="k", mew=0.4,
                    label=rf"{nb_min}-{nb_max}: $\times${np.median(y / x):.1f}")
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.plot([low, high], [low, high], "k--", lw=0.8)
        ax.set_xlim(low, high); ax.set_ylim(low, high); ax.set_aspect("equal")
        for axis in (ax.xaxis, ax.yaxis): axis.set_major_locator(LogLocator(numticks=10))   # a label at every decade
        ax.xaxis.set_minor_locator(LogLocator(subs=np.arange(2, 10), numticks=10)); ax.xaxis.set_minor_formatter(NullFormatter())
        ax.tick_params(axis="x", which="minor", length=2, width=0.5)
        ax.set_xlabel(r"$\gamma_{\rm GS2}$"); ax.set_ylabel(r"$\gamma_{\rm GFTM}$")
        fig.legend(title="NBASIS\n" + r"(median $\gamma_{\rm GFTM}/\gamma_{\rm GS2}$)", loc="outside right center")
        fig.savefig(plot_dir / f"fig_gftm_search_gamma_vs_gs2_pooled_filter{filter_value:g}_nbasis_overlay.pdf")
        plt.show()

# cell 11: single panel, pooled, coloured by log10(theta_code/theta_GS2)
for filter_value in gftm_filters.values():
    for nbasis in ((2, 4), (16, 16)):
        arm, reference_ = pooled_arm(filter_value, *nbasis), pooled_gs2
        found = (arm["gamma"] > 0) & np.isfinite(arm["extent"])
        x, y = reference_["gamma"][found], arm["gamma"][found]
        log_ratio = np.log10(arm["extent"][found] / reference_["extent"][found])
        print(f"{((x < low) | (y < low) | (x > high) | (y > high)).sum()} points outside axes")
        with plt.style.context([ROOT / "src/general_analysis/paper.mplstyle", ROOT.parent / "styles/paper.mplstyle"]):
            fig, ax = plt.subplots(figsize=(0.6 * 6.30, 0.6 * 6.30 * 0.85))
            sc = ax.scatter(x, y, s=10, alpha=0.8, c=log_ratio, vmin=-1, vmax=1, cmap="coolwarm")
            ax.set_xscale("log"); ax.set_yscale("log")
            ax.plot([low, high], [low, high], "k--", lw=0.8)
            ax.set_xlim(low, high); ax.set_ylim(low, high); ax.set_aspect("equal")
            ax.set_xlabel(r"$\gamma_{\rm GS2}$"); ax.set_ylabel(r"$\gamma_{\rm GFTM}$")
            fig.colorbar(sc, ax=ax, label=r"$\log_{10}(\theta_{\rm GFTM}/\theta_{\rm GS2})$")
            fig.savefig(plot_dir / f"fig_gftm_search_gamma_vs_gs2_colour_extent_ratio_pooled_filter{filter_value:g}_nbasis{nbasis[0]}-{nbasis[1]}.pdf")
            plt.show()

### Scatter grid (rows FILTER, columns NBASIS), the format of `fig_sat*_gamma_vs_gs2_pooled.pdf`

In [ ]:
case_colour = {"SPR-045": "tab:blue", "M1": "tab:orange", "R4": "tab:green", "R3": "tab:red"}
keys = [(filter_value, nmin, nmax) for filter_value in gftm_filters.values() for nmin, nmax in search_arms]
fig, axes = plt.subplots(2, 4, figsize=(20, 10), sharex=True, sharey=True, constrained_layout=True)
for ax, key in zip(axes.ravel(), keys):
    arm = pooled_arm(*key)
    found = (arm["gamma"] > 0) & np.isfinite(arm["extent"])
    stats = summary[(summary.kind == "search") & (summary.case == "pooled") & (summary["filter"] == key[0]) & (summary.nbasis_min == key[1])].iloc[0]
    x, y = pooled_gs2["gamma"][found], arm["gamma"][found]
    for case in cases:
        mask = pooled_case[found] == case
        ax.scatter(x[mask], y[mask], s=14, alpha=0.7, color=case_colour[case], label=f"{case} (n={mask.sum()})")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.text(0.97, 0.03, f"n={stats.n_found} (+{stats.n_miss} GFTM misses)\nabove $y=x$: {100*stats.frac_above:.0f}%\n"
            f"median $\\log_{{10}}(\\gamma_G/\\gamma_{{GS2}})$ = {stats.median_log_ratio:+.2f} [{stats.ci_lo:+.2f}, {stats.ci_hi:+.2f}]\n"
            f"mean ratio {stats.mean_ratio:.2f}\nLogMed $|\\gamma$ err$|$ = {stats.logmed_abs_err:.2f}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=10, bbox=dict(fc="white", alpha=0.85))
    ax.set_title(f"GFTM emulated search, FILTER {key[0]:g}, NBASIS {key[1]}-{key[2]}", fontsize=11)
for ax in axes[:, 0]: ax.set_ylabel(r"$\gamma_{\rm GFTM}$")
for ax in axes[-1]: ax.set_xlabel(r"$\gamma_{\rm GS2}$")
low_, high_ = min(min(ax.get_xlim()[0], ax.get_ylim()[0]) for ax in axes.ravel()), max(max(ax.get_xlim()[1], ax.get_ylim()[1]) for ax in axes.ravel())
for ax in axes.ravel():
    ax.plot([low_, high_], [low_, high_], "k--", lw=1); ax.set_xlim(low_, high_); ax.set_ylim(low_, high_); ax.set_aspect("equal")
axes[0, 0].legend(loc="upper left")
fig.savefig(plot_dir / "fig_gftm_search_gamma_vs_gs2_pooled.pdf", dpi=150, bbox_inches="tight"); plt.show()

### Summary: pooled median $r$ against NBASIS_MAX
GFTM emulated search (FILTER 0.5, 0), GFTM fixed $W=0.4$, TGLF SAT1/SAT2 search at FILTER 2 (read from `results/tglf_sat1_sat2/csv/bias_by_case_setting.csv`, not recomputed).

In [ ]:
tglf_bias = pd.read_csv(os.environ.get("TGLF_BIAS_CSV", "bias_by_case_setting_tglf.csv"))
tglf_bias = tglf_bias[(tglf_bias.case == "pooled") & (tglf_bias["filter"] == 2.0)]
with plt.style.context([ROOT / "src/general_analysis/paper.mplstyle", ROOT.parent / "styles/paper.mplstyle"]):
    fig, ax = plt.subplots()
    for kind, filter_value, style, label in (("search", 0.5, "-o", "GFTM search, FILTER 0.5"), ("search", 0.0, "--o", "GFTM search, FILTER 0"),
                                             ("fixed", 0.5, "-s", r"GFTM fixed $W=0.4$, FILTER 0.5"), ("fixed", 0.0, "--s", r"GFTM fixed $W=0.4$, FILTER 0")):
        rows_ = summary[(summary.kind == kind) & (summary.case == "pooled") & (summary["filter"] == filter_value)].sort_values("nbasis_max")
        ax.plot(rows_.nbasis_max, rows_.median_log_ratio, style, label=label)
        ax.fill_between(rows_.nbasis_max, rows_.ci_lo, rows_.ci_hi, alpha=0.15)
    for sat_rule, style in ((1, "k-^"), (2, "k--v")):
        rows_ = tglf_bias[tglf_bias.sat_rule == sat_rule].sort_values("nbasis_max")
        ax.plot(rows_.nbasis_max, rows_.median_log_ratio, style, mfc="none", label=f"TGLF SAT{sat_rule} search, FILTER 2")
    ax.axhline(0, color="0.5", lw=0.8)
    ax.set_xscale("log", base=2); ax.set_xlabel("NBASIS_MAX"); ax.set_ylabel(r"pooled median $\log_{10}(\gamma_{\rm code}/\gamma_{\rm GS2})$")
    ax.legend(fontsize=7, ncols=2)
    fig.savefig(plot_dir / "fig_summary_median_bias_vs_nbasis.pdf"); plt.show()

for pdf in plot_dir.glob("fig_*.pdf"):
    (copied_dir / pdf.name).write_bytes(pdf.read_bytes())